# Model Selection — NYC Taxi Trip Duration

`nyc-taxi select` picks the model to register from every run of `nyc-taxi train` (the RF baseline) and
`nyc-taxi tune` (20 Optuna trials), scores the winner **once** on the test days next to the baselines,
and registers it in MLflow **without an alias**.

**This notebook trains and computes nothing.** It reads the latest finished MLflow run `select`.
Before running: start MLflow (`docker compose up -d postgres mlflow`) and run `uv run nyc-taxi all`.

## 1. Selection rule (written before the results)

Data: train Jan 1–20 (every model is fitted on the same training sample of these days), validation
Jan 21–25 (comparing), test Jan 26–31 (used once).

Candidates: the RF baseline and every Optuna trial. The naive baselines are references, not candidates.

1. **Best:** the lowest validation RMSE.
2. **Tied:** every candidate whose validation RMSE is within **1%** of the best.
3. **Winner:** the **smallest model** (`model_mb`, serialized skops size) among the tied candidates;
   equal sizes are broken by validation RMSE.

The winner's settings are fitted again on the same training sample with the same seed (the same model),
scored once on test, and registered without an alias. The baselines are scored on the same test days.

In [ ]:
import io
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import mlflow
import pandas as pd
from IPython.display import Image, display

from src import selection, tracking
from src.experiment import TRACKING_URI
from src.train import MODEL_NAME

mlflow.set_tracking_uri(TRACKING_URI)
try:
    run = tracking.latest_run(selection.SELECT_RUN_NAME)
except LookupError as exc:
    raise SystemExit(str(exc)) from None  # no select run yet: run `uv run nyc-taxi all`
params, metrics, tags = run.data.params, run.data.metrics, run.data.tags
tolerance = float(params.get("rule_tolerance", selection.RMSE_TOLERANCE))


def artifact(name):
    """Local copy of one file logged in the select run."""
    return tracking.download_run_file(run.info.run_id, name)


def show(fig):
    """Display a matplotlib Figure as a PNG."""
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=110)
    display(Image(data=buffer.getvalue()))


print(f"select run {run.info.run_id} ({pd.Timestamp(run.info.start_time, unit='ms'):%Y-%m-%d %H:%M} UTC)")
print(artifact(selection.RULE_FILE).read_text(encoding="utf-8"))

## 2. Candidates within 1% of the best (tied)

In [ ]:
table = pd.read_csv(artifact(selection.CANDIDATES_FILE))
print(f"{int(table['tied'].sum())} of {len(table)} candidates tied within {tolerance:.0%} of the best validation RMSE")
columns = ["candidate", "winner", "val_rmse", "rmse_vs_best_pct", "val_mae", "model_mb", *selection.PARAM_KEYS]
table.loc[table["tied"], columns].round(4)

## 3. Validation RMSE vs model size

Left: every Random Forest candidate (grey) with the selected serving model highlighted in blue and
annotated; the dashed line is the RMSE tolerance (best validation RMSE + 1%), so the tied candidates
are the points below it. Right: the naive baselines' validation RMSE, as accuracy references.

In [ ]:
test_table = pd.read_csv(artifact(selection.TEST_FILE), index_col="model")
naive = {label: test_table.at[label, "val_rmse"] for label in selection.NAIVE_LABELS.values() if label in test_table.index}
show(selection.plot_rmse_vs_size(table, tolerance, naive))

## 4. Winner

In [ ]:
winner = table[table["winner"]].iloc[0]
print(f"Winner: {winner['candidate']}  {selection.candidate_params(winner)}")
print(f"  validation RMSE {winner['val_rmse']:.4f} ({winner['rmse_vs_best_pct']:+.2f}% vs best), "
      f"MAE {winner['val_mae']:.4f}, size {metrics.get('winner_model_mb', winner['model_mb']):.1f} MiB")
print(f"  test RMSE {metrics['test_rmse']:.4f}, test MAE {metrics['test_mae']:.4f} (scored once)")
print(f"  registered as {MODEL_NAME} version {tags.get('registered_version')} without an alias; "
      f"champion currently -> {tracking.alias_version(MODEL_NAME, 'champion')}")

## 5. Final test comparison (Jan 26–31, scored once)

Every row learned from the training days only; the naive baselines use training-day statistics and the
RF baseline is refitted on the same training sample.

In [ ]:
display(test_table[["val_rmse", "test_rmse", "test_mae"]].round(4))
ax = test_table["test_rmse"].plot.barh(figsize=(7, 3), title="Test RMSE (Jan 26-31, minutes)")
ax.invert_yaxis()
ax.figure.tight_layout()

**Next:** `uv run nyc-taxi verify <version>`, set the `champion` alias in the MLflow UI, then an
authenticated `/reload` (see `docs/operations.md`). The previous champion stays registered for rollback.